# Lesson 12.9 - Data-Centric Labeling Ops Workbench

## Learning Objectives
- Implement a toy active-learning loop.
- Build weak-supervision label functions and combine votes.
- Track labeling-efficiency metrics.

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1) Build Labeled/Unlabeled Pools

In [2]:
X, y = make_classification(n_samples=1200, n_features=14, n_informative=8, random_state=42)
idx = np.arange(len(X))
np.random.shuffle(idx)

seed_n = 120
labeled_idx = idx[:seed_n]
unlabeled_idx = idx[seed_n:900]
test_idx = idx[900:]

X_l, y_l = X[labeled_idx], y[labeled_idx]
X_u, y_u_true = X[unlabeled_idx], y[unlabeled_idx]
X_test, y_test = X[test_idx], y[test_idx]

print({"labeled": len(X_l), "unlabeled": len(X_u), "test": len(X_test)})

{'labeled': 120, 'unlabeled': 780, 'test': 300}


## 2) Baseline Model + Uncertainty Sampling

In [3]:
model = LogisticRegression(max_iter=500)
model.fit(X_l, y_l)
base_acc = accuracy_score(y_test, model.predict(X_test))

proba = model.predict_proba(X_u)[:, 1]
uncertainty = np.abs(proba - 0.5)
query_k = 80
query_idx = np.argsort(uncertainty)[:query_k]

print({"baseline_acc": round(base_acc, 4), "selected_for_labeling": int(query_k)})

{'baseline_acc': 0.7433, 'selected_for_labeling': 80}


## 3) Active Learning Update

In [4]:
X_new, y_new = X_u[query_idx], y_u_true[query_idx]

X_l2 = np.vstack([X_l, X_new])
y_l2 = np.concatenate([y_l, y_new])

model2 = LogisticRegression(max_iter=500)
model2.fit(X_l2, y_l2)
acc2 = accuracy_score(y_test, model2.predict(X_test))

print({"post_active_learning_acc": round(acc2, 4), "gain": round(acc2 - base_acc, 4)})

{'post_active_learning_acc': 0.78, 'gain': 0.0367}


## 4) Weak Supervision Label Functions (Toy)

In [5]:
pool_df = pd.DataFrame(X_u[:, :4], columns=["f1", "f2", "f3", "f4"])

# label functions return {0, 1, -1 abstain}
def lf_rule_1(row):
    return 1 if row.f1 + row.f2 > 1.2 else -1

def lf_rule_2(row):
    return 0 if row.f3 - row.f4 > 0.8 else -1

def lf_rule_3(row):
    return 1 if row.f1 > 1.0 else 0 if row.f1 < -1.0 else -1

lf_outputs = np.array([[lf_rule_1(r), lf_rule_2(r), lf_rule_3(r)] for r in pool_df.itertuples()])

def majority_vote(labels):
    vals = [x for x in labels if x != -1]
    if not vals:
        return -1
    return int(np.mean(vals) >= 0.5)

weak_labels = np.array([majority_vote(row) for row in lf_outputs])
coverage = float(np.mean(weak_labels != -1))
print({"weak_label_coverage": round(coverage, 4)})

{'weak_label_coverage': 0.9064}


## 5) Labeling Ops KPIs

In [6]:
kpi = pd.DataFrame([
    {"metric": "baseline_accuracy", "value": base_acc},
    {"metric": "active_learning_accuracy", "value": acc2},
    {"metric": "accuracy_gain", "value": acc2 - base_acc},
    {"metric": "weak_label_coverage", "value": coverage},
])
kpi

,metric,value
0,baseline_accuracy,0.743333
1,active_learning_accuracy,0.780000
2,accuracy_gain,0.036667
3,weak_label_coverage,0.906410


## Connect to Theory
- Active learning prioritizes high-value annotation spend.
- Weak supervision bootstraps labels fast but requires QA controls.
- KPI tracking links data ops to model outcomes.

## Case Studies & Exceptions
1. Support intent labeling: uncertainty sampling reduced annotation cost.
2. Compliance classification: weak labels accelerated initial dataset creation.
3. Exception: unstable label taxonomy can negate active-learning gains.

## Interview Questions & Answers
1. **Q:** Why data-centric labeling ops? **A:** Data quality often limits model performance more than architecture.
2. **Q:** What is active learning? **A:** Selecting most informative samples for manual labeling.
3. **Q:** Uncertainty sampling intuition? **A:** Label points where model confidence is low.
4. **Q:** What is weak supervision? **A:** Programmatic noisy-label generation from heuristics.
5. **Q:** Why monitor coverage in weak supervision? **A:** To understand how much of pool gets label signal.
6. **Q:** Major weak-supervision risk? **A:** Systematic heuristic bias.
7. **Q:** Why measure accuracy gain per batch? **A:** To quantify annotation ROI.
8. **Q:** What is disagreement review? **A:** Human adjudication on conflicting labels/policies.
9. **Q:** How prevent leakage? **A:** Strict split management and dedup checks.
10. **Q:** One-line guidance? **A:** Treat labeling as an engineering system with metrics, QA, and governance.